# Video to 3D Scene Restructured
This notebook allows you to train a 3D scene using Gaussian Splatting from either a test dataset or your own video.

In [ ]:

!nvidia-smi
!python -m pip install -q -U setuptools wheel ninja
!python -m pip install -q nerfstudio gsplat huggingface_hub pillow plyfile
!rm -rf /content/video2scene_colab

Sat May 23 15:58:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from PIL import Image
import torch
import gsplat
from huggingface_hub import snapshot_download, hf_hub_download

# --- CONFIGURATION ---
# If VIDEO_PATH is empty, the default 'poster' test example dataset will be downloaded.
VIDEO_PATH = "" # @param {type:"string"}
METHOD_NAME = "splatfacto-big" # @param ["splatfacto", "splatfacto-big"]
MAX_ITERATIONS = 15000 # @param {type:"integer"}

os.environ["MAX_JOBS"] = "1"
os.environ["CMAKE_BUILD_PARALLEL_LEVEL"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["TORCH_EXTENSIONS_DIR"] = "/content/torch_extensions"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

root = Path("/content/video2scene_colab")
root.mkdir(parents=True, exist_ok=True)

print(f"Using device: {torch.cuda.get_device_name(0)}")

Using device: Tesla T4


In [ ]:
def extract_frames(video_input, output_dir, fps=5):
    """Extracts frames from a video file using ffmpeg."""
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting frames from {video_input} at {fps} FPS...")

    subprocess.run([
        "ffmpeg", "-i", str(video_input),
        "-vf", f"fps={fps}",
        "-q:v", "2",
        str(output_dir / "frame_%05d.png")
    ], check=True)

    return sorted(output_dir.glob("*.png"))


def prepare_data():
    data_dir = root / "data_source"
    images_dir = data_dir / "images"
    shutil.rmtree(data_dir, ignore_errors=True)
    images_dir.mkdir(parents=True)

    if VIDEO_PATH and os.path.exists(VIDEO_PATH):
        extract_frames(Path(VIDEO_PATH), images_dir)
        raise RuntimeError(
            "Custom videos need camera poses before Splatfacto training. "
            "Run Nerfstudio/COLMAP preprocessing first, or leave VIDEO_PATH empty "
            "to use the prepared poster example."
        )

    print("No video provided. Downloading prepared poster dataset...")
    snapshot_dir = root / "hf_snapshot"
    shutil.rmtree(snapshot_dir, ignore_errors=True)

    snapshot_download(
        repo_id="nerfstudioteam/datasets",
        repo_type="dataset",
        allow_patterns=["poster/images_2/*.png"],
        local_dir=snapshot_dir,
        local_dir_use_symlinks=False,
    )

    source_images = sorted((snapshot_dir / "poster" / "images_2").glob("*.png"))
    assert source_images, "No poster images were downloaded."

    for image_path in source_images:
        shutil.copy2(image_path, images_dir / image_path.name)

    transforms_path = Path(hf_hub_download(
        repo_id="nerfstudioteam/datasets",
        repo_type="dataset",
        filename="poster/transforms.json",
    ))
    sparse_pc_path = Path(hf_hub_download(
        repo_id="nerfstudioteam/datasets",
        repo_type="dataset",
        filename="poster/sparse_pc.ply",
    ))
    shutil.copy2(sparse_pc_path, data_dir / "sparse_pc.ply")

    meta = json.loads(transforms_path.read_text())
    available_images = {path.name for path in source_images}

    meta["frames"] = [
        {**frame, "file_path": f"./images/{Path(frame['file_path']).name}"}
        for frame in meta["frames"]
        if Path(frame["file_path"]).name in available_images
    ]
    assert meta["frames"], "No frames in transforms.json matched downloaded images."

    with Image.open(source_images[0]) as image:
        width, height = image.size

    scale_x = meta["w"] / width
    scale_y = meta["h"] / height

    for key in ("fl_x", "cx"):
        meta[key] /= scale_x
    for key in ("fl_y", "cy"):
        meta[key] /= scale_y

    meta["w"] = width
    meta["h"] = height
    meta["ply_file_path"] = "sparse_pc.ply"

    (data_dir / "transforms.json").write_text(json.dumps(meta, indent=2))

    print("data_dir:", data_dir)
    print("frames:", len(meta["frames"]))
    print("image size:", width, height)
    print("first frame:", meta["frames"][0]["file_path"])
    return data_dir


shutil.rmtree(root / "outputs", ignore_errors=True)
working_data = prepare_data()


No video provided. Downloading test dataset...


Fetching ... files: 0it [00:00, ?it/s]

Metadata prepared with 100 sequential frames (frame_00001 to frame_00100).


In [ ]:
output_dir = root / "outputs"
log_path = root / "train.log"
shutil.rmtree(output_dir, ignore_errors=True)

print("Starting Training...")
!ns-train {METHOD_NAME} \
  --output-dir "{output_dir}" \
  --experiment-name experiment_1 \
  --max-num-iterations {MAX_ITERATIONS} \
  --steps-per-save 1000 \
  --vis tensorboard \
  --data "{working_data}" \
  > "{log_path}" 2>&1 || (echo "--- TRAINING FAILED. LOG CONTENT: ---" && cat "{log_path}")

print(f"\nTraining session ended. Check {log_path} for details.")


Starting Training...

Training session ended. Check /content/video2scene_colab/train.log for details.


In [ ]:
config_files = sorted(output_dir.glob("experiment_1/*/*/config.yml"))
print("configs:", len(config_files))

valid_runs = []
for config in config_files:
    checkpoints = sorted((config.parent / "nerfstudio_models").glob("*.ckpt"))
    print(config)
    print("  checkpoints:", len(checkpoints))
    if checkpoints:
        print("  latest:", checkpoints[-1])
        valid_runs.append((config, checkpoints[-1]))

config_path, latest_checkpoint = valid_runs[-1]

export_dir = root / "exports" / "splat_output"
shutil.rmtree(export_dir, ignore_errors=True)

!ns-export gaussian-splat \
  --load-config "{config_path}" \
  --output-dir "{export_dir}"

print(f"Exported to: {export_dir}")
for path in sorted(export_dir.rglob("*")):
    if path.is_file():
        print(path, round(path.stat().st_size / 1024 / 1024, 2), "MB")


2026-05-23 17:03:48.874092: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.12/dist-packages/torchmetrics/functional/image/lpips.py:332: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`we